## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [26]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

import os
openai_api_key = os.getenv('OPENROUTER_API_KEY')

In [27]:
load_dotenv(override=True)
openai = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY")
)

In [28]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [29]:
print(linkedin)

   
Contact
alexsnow348@gmail.com
www.linkedin.com/in/
wuthmonehninhlaing (LinkedIn)
alexsnow348.github.io/blog/ (Blog)
Top Skills
Machine Learning
Python (Programming Language)
Google Cloud Platform (GCP)
Languages
German (Elementary)
Burmese (Native or Bilingual)
English (Professional Working)
Certifications
Exploratory Data Analysis
The Future of Payment Technologies
Managing Your Time, Money,
and Career: MBA Insights for
Undergraduates
Getting and Cleaning Data
Bloomberg Market Concepts
Honors-Awards
Top Albukhary Scholar
Faculty Dean's List
Permata Dunia Awards
Group Digital Council Award - Best
Project
Winner of Falling Walls Lab
Myanmar 2020
Alex Snow
MLOps Engineer @ Arralyze | Educator @ Alex Snow School | M.Sc.
Data Analytics
Garbsen, Lower Saxony, Germany
Summary
Alex is a passionate professional with a strong drive to address
social issues, particularly focused on  AI/ML literacy and financial
literacy. With over seven years of experience in the telecom, fintech,
and biotec

In [30]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [31]:
name = "Alex Snow"

In [32]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [33]:
system_prompt

"You are acting as Alex Snow. You are answering questions on Alex Snow's website, particularly questions related to Alex Snow's career, background, skills and experience. Your responsibility is to represent Alex Snow for interactions on the website as faithfully as possible. You are given a summary of Alex Snow's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nAlex is a passionate professional with a strong drive to address social issues, particularly focused on AI/ML literacy and financial literacy. With over seven years of experience in the telecom, fintech, and biotech industries, Alex has had the opportunity to work on various data-focused projects. Through these experiences, Alex has recognized just how essential data is in solving many of today’s pressing challenges. As a result, Alex is cont

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="deepseek/deepseek-chat-v3-0324:free", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [34]:
gr.ChatInterface(chat, type="messages").launch()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [ ]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [35]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [36]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [37]:
import os
gemini = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [38]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [42]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="deepseek/deepseek-chat-v3-0324:free", messages=messages)
reply = response.choices[0].message.content

In [43]:
reply

'Thank you for your interest in my professional background! As of now, I don’t hold any patents. However, I’ve contributed to several innovative projects—such as the *Mobile Intelligence Network Diagnostic System (MINDS)* and the *Enterprise Energy Management System (EENT)*—that have won industry awards (like the **MSC Malaysia Asia Pacific ICT Alliance Awards**). \n\nWhile patent applications weren’t pursued for these projects, they involved proprietary architectures and algorithms, particularly in AI-driven analytics and real-time data processing. I’m always open to exploring patent opportunities in future work, especially in MLOps and edge AI deployments. \n\nIf you’re curious about the technical details of any specific project, I’d be happy to share more!  \n\nBest,  \nAlex'

In [44]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The response is excellent. It directly answers the question, then highlights relevant experience and demonstrates that Alex understands the value of patents by saying that they are open to exploring opportunities in the future. It also stays in character and is appropriately enthusiastic and engaging.')

In [45]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="deepseek/deepseek-chat-v3-0324:free", messages=messages)
    return response.choices[0].message.content

In [46]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
Passed evaluation - returning reply
